<a href="https://colab.research.google.com/github/FishyFoshy/COMP3608-Project/blob/main/Dataset_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Cleaning of Dataset 3

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)

# Load datasets
df3 = pd.read_csv('Dataset 3.csv')

# Drops all unnecessary columns
df3 = df3.drop(columns=['id', 'date', 'yr_renovated', 'zipcode', 'sqft_living15', 'sqft_lot15'])

# Drops all with null values
df3.dropna(inplace=True)

# Remove the top 1% highest prices to reduce high-end outliers before log-transform
upper_quantile = df3['price'].quantile(0.99)
df3 = df3[df3['price'] <= upper_quantile]

# Apply log transform to the target
df3['price'] = np.log(df3['price'])

# Create a copy of df3 for Linear Regression before feature engineering
df3lr = df3.copy()

# Apply location-based feature engineering to df3lr only (for Linear Regression)
df3lr['lat_lon_interaction'] = df3lr['lat'] * df3lr['long']
center_lat = df3lr['lat'].mean()
center_lon = df3lr['long'].mean()
df3lr['dist_to_center'] = np.sqrt((df3lr['lat'] - center_lat)**2 + (df3lr['long'] - center_lon)**2)
df3lr = df3lr.drop(columns=['lat', 'long'])  # Drop original lat/lon to avoid multicollinearity

# Separates non numerical columns from numerical ones
non_numerical_columns = ['price']

# Process df3lr (for Linear Regression): scale and encode
numerical_features_lr = [col for col in df3lr.columns if col not in non_numerical_columns]
scaler_lr = StandardScaler()
df3lr[numerical_features_lr] = scaler_lr.fit_transform(df3lr[numerical_features_lr])
categorical_cols_lr = df3lr.select_dtypes(include='object').columns
df3lr = pd.get_dummies(df3lr, columns=categorical_cols_lr, drop_first=True)

target_column = 'price'
x_lr = df3lr[[col for col in df3lr.columns if col != target_column]].copy()
y_lr = df3lr[target_column].copy()
x_train_lr, x_test_lr, y_train_lr, y_test_lr = train_test_split(x_lr, y_lr, test_size=0.2, random_state=42)

# Process df3 (for tree models: Random Forest, XGBoost): keep raw lat/lon, scale and encode
numerical_features = [col for col in df3.columns if col not in non_numerical_columns]
scaler = StandardScaler()
df3[numerical_features] = scaler.fit_transform(df3[numerical_features])
categorical_cols_to_encode = df3.select_dtypes(include='object').columns
df3 = pd.get_dummies(df3, columns=categorical_cols_to_encode, drop_first=True)

features = [col for col in df3.columns if col != target_column]
x = df3[features].copy()
y = df3[target_column].copy()
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

print("=" * 60)
print("DATASET 3: Dataset 3.csv")
print("=" * 60)
print("\nFirst 10 rows:")
display(df3.head(10))
print("\nData types:")
print(df3.dtypes)
print("\nMissing values:")
print(df3.isnull().sum())
print("\nTarget variable (price) statistics:")
print(df3['price'].describe())

In [ ]:
from sklearn.metrics import make_scorer, mean_squared_error, mean_absolute_error, r2_score
import numpy as np
# Inverse the log transformation to its original scale

def rmse_original_scale(y_true, y_pred):
    y_true_original = np.exp(y_true)
    y_pred_original = np.exp(y_pred)
    # Calculate RMSE on the original scale
    return np.sqrt(mean_squared_error(y_true_original, y_pred_original))

def mae_original_scale(y_true, y_pred):
    y_true_original = np.exp(y_true)
    y_pred_original = np.exp(y_pred)
    # Calculate MAE on the original scale
    return mean_absolute_error(y_true_original, y_pred_original)

def r2_original_scale(y_true, y_pred):
    y_true_original = np.exp(y_true)
    y_pred_original = np.exp(y_pred)
    # Calculate R2 on the original scale
    return r2_score(y_true_original, y_pred_original)

# Create scorers that can be used with cross_val_score
neg_rmse_original_scorer = make_scorer(rmse_original_scale, greater_is_better=False)
neg_mae_original_scorer = make_scorer(mae_original_scale, greater_is_better=False)
r2_original_scorer = make_scorer(r2_original_scale, greater_is_better=True)

### Linear Regression for Dataset 3

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np
import matplotlib.pyplot as plt

model = LinearRegression()
model.fit(x_train_lr, y_train_lr)
predictions = model.predict(x_test_lr)

# Inverse log transform to get actual prices
y_test_lr_actual = np.exp(y_test_lr)
predictions_actual = np.exp(predictions)

# Evaluate on original scale
rmse_lr = np.sqrt(mean_squared_error(y_test_lr_actual, predictions_actual))
mae_lr = mean_absolute_error(y_test_lr_actual, predictions_actual)
r2_lr = r2_score(y_test_lr_actual, predictions_actual)

print("=" * 50)
print("BASELINE: Linear Regression on Dataset 3.csv")
print("=" * 50)
print(f"RMSE:\t${rmse_lr:,.2f}")
print(f"MAE:\t${mae_lr:,.2f}")
print(f"R²:\t{r2_lr:.4f}")
print(f"\nCross-validation RMSE:\t${-cross_val_score(model, x_lr, y_lr, cv=5, scoring=neg_rmse_original_scorer).mean():,.2f}")
print(f"Cross-validation MAE:\t${-cross_val_score(model, x_lr, y_lr, cv=5, scoring=neg_mae_original_scorer).mean():,.2f}")
print(f"Cross-validation R²:\t{cross_val_score(model, x_lr, y_lr, cv=5, scoring=r2_original_scorer).mean():.4f}")

# Predicted vs Actual scatter
plt.figure(figsize=(8, 6))
plt.scatter(y_test_lr_actual, predictions_actual, alpha=0.4, s=15)
plt.plot([0, y_test_lr_actual.max()], [0, y_test_lr_actual.max()], 'r--', label='Perfect prediction')
plt.xlabel('Actual Price ($)')
plt.ylabel('Predicted Price ($)')
plt.title('Baseline Linear Regression: Predicted vs Actual House Price')
plt.legend()
plt.tight_layout()
plt.show()

before cross validaion ,from our results, the mean square error was very large approximately 46 billion, which when working with prices of hunderds of thoursand which are then squared can happen making this less interpretable as it represents dollars.

However the root mean square error, sq_root(mse), explains that our model was off by $215 172 on averge when prdicting houses but when squaring the values they are susceptible to outliars, which the dataset has, visible in the scatter plot.

the r_sq value of 0.69 showed that our model predicts 69% of the variance of the dataset

 after cross_validation, we can see a decrese in results as the mean square error was reduced t0 approximately 40 billion, the root mean square error, was reduced to approximately 200 000, meaning there was a decrease in how much our prediction were off by but the r2 value was till approximately 0.69, meaning our model still represents 69% of the variance of the dataset.

from these results we can conclude the model predicted fairly .

limitation : the inclusion of outliars